# Day 46: Watch an AI Agent Think

Yesterday we graded an AI's *answers*. But an **agent** doesn't just answer — it plans, calls tools, reads the results, and tries again. If you only see the final reply, you have no idea *how* it got there.

Today we **trace** a real agent: every step it takes, every tool it calls, every result it gets back. Seeing the full trajectory is the foundation of agent evaluation — **you can't grade what you can't see.**

## Install

In [ ]:
%pip install langchain-google-genai langchain --quiet

## Setup — An Agent With Two Tools

We give the agent a **lookup** tool (country facts) and a **calculator**. The task we'll give it needs *both* — and in the right order. That's what makes the trace interesting.

In [5]:
import os, ast, operator
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"].strip()

# Safe arithmetic — walk the AST, allow ONLY numbers + math operators.
# (Never use eval() for this: it would run arbitrary code the model emits.)
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
        ast.Mod: operator.mod, ast.Pow: operator.pow,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _safe_math(node):
    if isinstance(node, ast.Expression):
        return _safe_math(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_math(node.left), _safe_math(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_math(node.operand))
    raise ValueError("unsupported expression")

@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression, e.g. '123.8 - 68.2'."""
    try:
        return str(_safe_math(ast.parse(expression, mode="eval")))
    except Exception as e:
        return f"error: {e}"

@tool
def lookup(country: str) -> str:
    """Look up a country's capital and population (in millions)."""
    db = {
        "japan":  "Tokyo; population 123.8 million",
        "france": "Paris; population 68.2 million",
        "brazil": "Brasilia; population 216 million",
    }
    return db.get(country.strip().lower(), "not found")

tools = [calculator, lookup]
by_name = {t.name: t for t in tools}
agent_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0).bind_tools(tools)
print("✅ Agent ready — tools: lookup, calculator")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Agent ready — tools: lookup, calculator


## One Task That Needs a Plan

> *"Which country has more people, Japan or France — and how many more, in millions?"*

A single question, but the agent can't answer it in one shot. It has to look up **two** countries, then **subtract**. Watch it figure that out.

## The Trace — Every Step, Every Tool, Every Result

We run the agent loop and print the whole trajectory: each **ACT** (the tool it chose + the arguments) and each **OBSERVE** (what came back). This is the agent's actual chain of actions — not a summary.

In [6]:
def trace_agent(task, max_steps=6):
    print("=" * 60)
    print("TASK:", task)
    print("=" * 60)
    msgs = [HumanMessage(content=task)]
    step = 0
    trajectory = []
    for _ in range(max_steps):
        ai = agent_model.invoke(msgs)
        msgs.append(ai)
        if not ai.tool_calls:
            text = ai.content if isinstance(ai.content, str) else ai.content[0]["text"]
            print("\n🏁 FINAL ANSWER")
            print("   " + text.strip())
            return trajectory, text.strip()
        for call in ai.tool_calls:
            step += 1
            result = by_name[call["name"]].invoke(call["args"])
            trajectory.append(call["name"])
            print(f"\n🔹 STEP {step}")
            print(f"   ACT      → {call['name']}({call['args']})")
            print(f"   OBSERVE  ← {result}")
            msgs.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return trajectory, "(stopped: hit max steps)"

trajectory, answer = trace_agent(
    "Which country has more people, Japan or France, and how many more (in millions)?"
)

TASK: Which country has more people, Japan or France, and how many more (in millions)?

🔹 STEP 1
   ACT      → lookup({'country': 'Japan'})
   OBSERVE  ← Tokyo; population 123.8 million

🔹 STEP 2
   ACT      → lookup({'country': 'France'})
   OBSERVE  ← Paris; population 68.2 million

🔹 STEP 3
   ACT      → calculator({'expression': '123.8 - 68.2'})
   OBSERVE  ← 55.599999999999994

🏁 FINAL ANSWER
   Japan has more people than France. Japan has 55.6 million more people than France.


## Read the Plan Off the Trace

The agent was never *told* the steps. From the trajectory you can see the plan it formed on its own:

1. **lookup(Japan)** → 123.8 million
2. **lookup(France)** → 68.2 million  *(it fired both lookups together — they don't depend on each other)*
3. **calculator(123.8 - 68.2)** → 55.6
4. **synthesise** → "Japan, by 55.6 million"

That chain — **pick a tool → read the result → decide the next move → finish** — is the agent *reasoning*. The trace is the only place you can actually watch it happen.

In [ ]:
print("Tools the agent called, in order:")
print("   " + " → ".join(trajectory))
print(f"\nSteps taken: {len(trajectory)}")
print(f"Final answer: {answer}")

## Why the Trace Matters (this is where evals begin)

If you only saw *"Japan, by 55.6 million"*, you'd have to trust it. The trace lets you **verify the process**, not just the answer:

- Did it call the **right tools**? ✅ lookup for facts, calculator for math
- Did it use the **right arguments**? ✅ the two countries, the correct subtraction
- Did it **finish the task**? ✅

A right answer reached the *wrong* way (guessing the math instead of using the calculator) is just luck — and only the trace reveals it. **Grading that trajectory, at scale, across many runs, is exactly what "agent evals" means** — and tools like LangSmith visualise these traces for you automatically.

## Key Takeaways

1. **An agent is a loop, not a reply** — plan → act (tool) → observe → repeat → answer. The trace is that loop made visible.
2. **You can't grade what you can't see** — tracing the trajectory (which tools, which args, in what order) is the foundation every agent eval is built on.
3. **Verify the path, not just the destination** — a correct answer via the wrong steps is luck; the trace is how you catch it.

---

Next: Day 47 — Coming soon